# E5 — DistilBERT

**E5 — DistilBERT fine-tune (transfer learning). PyTorch/HuggingFace, GPU-accelerated if available.**

In [1]:
import os, re, time, json, pickle
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = "../data/IMDB Dataset.csv"
RESULTS_DIR = "../results"
TOKENIZER_PATH = "../results/tokenizer.pkl"
VOCAB_SIZE = 10000
EMBED_DIM = 100
SAMPLE_SIZE = 15000   # <-- subsample for faster training

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["review"] = df["review"].apply(lambda t: re.sub(r"<br\s*/?>", " ", str(t)))
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
assert df["label"].isna().sum() == 0, "Unexpected sentiment values — check the column."

# Subsample BEFORE splitting, so train/test shrink together and stay balanced
df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

X = df["review"].astype(str).to_numpy()
y = df["label"].to_numpy(dtype=int)

from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")

Train: 12000  Test: 3000


In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

torch.manual_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3

train_ds = Dataset.from_pandas(pd.DataFrame({"review": X_train_text, "label": y_train}))
test_ds = Dataset.from_pandas(pd.DataFrame({"review": X_test_text, "label": y_test}))

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return bert_tokenizer(batch["review"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True).rename_column("label", "labels")
test_ds = test_ds.map(tokenize, batched=True).rename_column("label", "labels")
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

/home/manikya/nlp_project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 139.20it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    auc = roc_auc_score(labels, probs)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc}

args = TrainingArguments(
    output_dir="../results/distilbert_out",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, num_train_epochs=EPOCHS,
    weight_decay=0.01, load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=100, fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=bert_model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

start = time.time()
trainer.train()
train_time = time.time() - start
metrics = trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.261478,0.281093,0.890333,0.920736,0.857049,0.887752,0.964072
2,0.196193,0.316386,0.892667,0.869592,0.926877,0.897321,0.964806
3,0.101604,0.403866,0.889000,0.876669,0.908432,0.892268,0.963081


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.101604,0.316386,3,0.892667,0.869592,0.926877,0.897321,0.964806


In [4]:
row = {
    "run_name": "E5_distilbert",
    "accuracy": round(metrics["eval_accuracy"], 4),
    "precision": round(metrics["eval_precision"], 4),
    "recall": round(metrics["eval_recall"], 4),
    "f1": round(metrics["eval_f1"], 4),
    "roc_auc": round(metrics["eval_roc_auc"], 4),
    "train_time_sec": round(train_time, 1),
    "model": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "E5_distilbert.csv")
pd.DataFrame([row]).to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(json.dumps(row, indent=2))

Saved ../results/E5_distilbert.csv
{
  "run_name": "E5_distilbert",
  "accuracy": 0.8927,
  "precision": 0.8696,
  "recall": 0.9269,
  "f1": 0.8973,
  "roc_auc": 0.9648,
  "train_time_sec": 393.8,
  "model": "distilbert-base-uncased",
  "max_len": 256,
  "epochs": 3
}
